# Notebook 1: Build the Raw Combined Dataset
### Smart City Bus Clustering — Greater Manchester Bee Network

Parses raw SIRI-VM location XML, TransXChange timetable XML, and the NeTEx-derived fares and disruption catalogues, then merges them into a single combined dataset.

*Cleaned for presentation: dead-end exploration and superseded retries removed, keeping the cells that show real pipeline logic and the two debugging stories (the join fan-out, and the disruption flag fix).*

## Step 1: Environment Setup & Data Paths

In [1]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession


from pyspark.sql.types import StructType, StructField, StringType, DoubleType
import  glob

spark = SparkSession.builder \
    .appName("SmartCityBusClustering") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

sc = spark.sparkContext
print("Spark version:", spark.version)
print("Default parallelism:", sc.defaultParallelism)

PROJECT_ROOT = "/Users/aayushbohara/Desktop/smartcity-bus-clustering"
PATHS = {
    "location":    os.path.join(PROJECT_ROOT, "data/raw/location"),
    "timetable":   os.path.join(PROJECT_ROOT, "data/raw/Timestabledata"),
    "fares":       os.path.join(PROJECT_ROOT, "data/raw/Faresdata"),
    "disruption":  os.path.join(PROJECT_ROOT, "data/raw/TFGM_Disruption"),
}

for name, p in PATHS.items():
    print(name, "->", p, "exists:", os.path.exists(p))

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/23 07:35:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/07/23 07:35:06 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark version: 4.1.1
Default parallelism: 10
location -> /Users/aayushbohara/Desktop/smartcity-bus-clustering/data/raw/location exists: True
timetable -> /Users/aayushbohara/Desktop/smartcity-bus-clustering/data/raw/Timestabledata exists: True
fares -> /Users/aayushbohara/Desktop/smartcity-bus-clustering/data/raw/Faresdata exists: True
disruption -> /Users/aayushbohara/Desktop/smartcity-bus-clustering/data/raw/TFGM_Disruption exists: True


## Step 2: File Discovery & Sanity Check

In [2]:
def count_files(folder, ext="*.xml", recursive=True):
    pattern = os.path.join(folder, "**", ext) if recursive else os.path.join(folder, ext)
    files = glob.glob(pattern, recursive=recursive)
    return files

location_files   = count_files(PATHS["location"])
timetable_files  = count_files(PATHS["timetable"])
fares_files      = count_files(PATHS["fares"])

print("Location XML files:", len(location_files))
print("Timetable XML files:", len(timetable_files))
print("Fares XML files:", len(fares_files))

# rough total size on disk, helps judge parsing time
def total_size_mb(files):
    return sum(os.path.getsize(f) for f in files) / (1024*1024)

print(f"Location total size: {total_size_mb(location_files):.1f} MB")
print(f"Timetable total size: {total_size_mb(timetable_files):.1f} MB")
print(f"Fares total size: {total_size_mb(fares_files):.1f} MB")

# disruption folder is CSV catalogues, not XML — list separately
disruption_files = glob.glob(os.path.join(PATHS["disruption"], "*.csv"))
print("Disruption CSV files:", disruption_files)

Location XML files: 345
Timetable XML files: 1318
Fares XML files: 4345
Location total size: 98.7 MB
Timetable total size: 569.8 MB
Fares total size: 2838.7 MB
Disruption CSV files: ['/Users/aayushbohara/Desktop/smartcity-bus-clustering/data/raw/TFGM_Disruption/location_data_catalogue.csv', '/Users/aayushbohara/Desktop/smartcity-bus-clustering/data/raw/TFGM_Disruption/fares_data_catalogue.csv', '/Users/aayushbohara/Desktop/smartcity-bus-clustering/data/raw/TFGM_Disruption/overall_compliance_report.csv', '/Users/aayushbohara/Desktop/smartcity-bus-clustering/data/raw/TFGM_Disruption/organisations_data_catalogue.csv', '/Users/aayushbohara/Desktop/smartcity-bus-clustering/data/raw/TFGM_Disruption/overall_data_catalogue.csv', '/Users/aayushbohara/Desktop/smartcity-bus-clustering/data/raw/TFGM_Disruption/timetables_data_catalogue.csv', '/Users/aayushbohara/Desktop/smartcity-bus-clustering/data/raw/TFGM_Disruption/operator_noc_data_catalogue.csv', '/Users/aayushbohara/Desktop/smartcity-bus-cl

## Step 3: Parse Location (SIRI-VM) XML — Distributed

File list is distributed across partitions with `sc.parallelize` and parsed with `mapPartitions`, satisfying the distributed-processing requirement.

In [8]:
import glob

file_list = glob.glob(os.path.join(PATHS["location"], "*.xml"))
print(f"{len(file_list)} location files found")

# Distribute the FILE LIST across partitions — satisfies "distributed processing" requirement
files_rdd = sc.parallelize(file_list, numSlices=8)

def parse_location_file(iterator):
    rows = []
    for path in iterator:
        try:
            tree = ET.parse(path)
            root = tree.getroot()
        except ET.ParseError:
            continue  # skip broken files (like the old 6-byte ones, if any remain)

        for va in root.iter():
            if va.tag.endswith("VehicleActivity"):
                def find(tag):
                    for e in va.iter():
                        if e.tag.endswith(tag):
                            return e.text
                    return None

                rows.append((
                    find("RecordedAtTime"),
                    find("LineRef"),
                    find("DirectionRef"),
                    find("VehicleRef"),
                    find("Latitude"),
                    find("Longitude"),
                    os.path.basename(path)
                ))
    return rows

import xml.etree.ElementTree as ET  # ensure imported in this cell too

parsed_rdd = files_rdd.mapPartitions(parse_location_file)
parsed_rdd.cache()

row_count = parsed_rdd.count()
print("Parsed location rows:", row_count)
print("Sample row:", parsed_rdd.take(1))

345 location files found


[Stage 0:>                                                          (0 + 8) / 8]

Parsed location rows: 79859
Sample row: [('2026-07-22T14:47:43+00:00', '23', 'inbound', '10042', '53.446323', '-2.310226', 'feed_14336_20260722_203501.xml')]


## Step 4: Build the Location DataFrame

Casts types, repartitions by join key (`lineRef`), and caches.

In [10]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

schema = StructType([
    StructField("timestamp", StringType()),
    StructField("lineRef", StringType()),
    StructField("directionRef", StringType()),
    StructField("vehicleRef", StringType()),
    StructField("latitude", StringType()),
    StructField("longitude", StringType()),
    StructField("source_file", StringType()),
])

location_df = spark.createDataFrame(parsed_rdd, schema=schema)

# cast lat/long to proper doubles, and timestamp to a real timestamp type
from pyspark.sql.functions import col, to_timestamp

location_df = location_df \
    .withColumn("latitude", col("latitude").cast("double")) \
    .withColumn("longitude", col("longitude").cast("double")) \
    .withColumn("timestamp", to_timestamp(col("timestamp")))

# repartition by join key (satisfies repartitioning requirement + real perf benefit for the upcoming join)
location_df = location_df.repartition(8, "lineRef")
location_df.cache()

print("Row count:", location_df.count())
print("Partitions:", location_df.rdd.getNumPartitions())
location_df.printSchema()
location_df.show(5, truncate=False)

Row count: 79859
Partitions: 8
root
 |-- timestamp: timestamp (nullable = true)
 |-- lineRef: string (nullable = true)
 |-- directionRef: string (nullable = true)
 |-- vehicleRef: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- source_file: string (nullable = true)

+-------------------+-------+------------+----------+---------+---------+------------------------------+
|timestamp          |lineRef|directionRef|vehicleRef|latitude |longitude|source_file                   |
+-------------------+-------+------------+----------+---------+---------+------------------------------+
|2026-07-22 20:28:47|191    |inbound     |10443     |53.478527|-2.237481|feed_14336_20260722_203501.xml|
|2026-07-22 20:34:56|191    |inbound     |10448     |53.410716|-2.164442|feed_14336_20260722_203501.xml|
|2026-07-22 20:33:54|149    |outbound    |10851     |53.521477|-2.213849|feed_14336_20260722_203501.xml|
|2026-07-22 20:34:39|149    |outbound   

## Step 5: Inspect Timetable (TransXChange) XML Tags

One sample file is inspected first to find the correct tag names before writing the full parser.

In [11]:
import glob

timetable_files = glob.glob(os.path.join(PATHS["timetable"], "**", "*.xml"), recursive=True)
print("Total timetable files:", len(timetable_files))

# peek one file's structure so we can extract the right tags
sample_file = timetable_files[0]
tree = ET.parse(sample_file)
root = tree.getroot()

# print unique tag names (namespace-stripped) found in this file, gives us a map of what's available
tags_seen = set()
for elem in root.iter():
    tag = elem.tag.split('}')[-1]  # strip namespace
    tags_seen.add(tag)

print(f"\nSample file: {os.path.basename(sample_file)}")
print("Unique tags found:", sorted(tags_seen))

Total timetable files: 1318

Sample file: BNSM_875_BNSMPC000368118010215875_20250831_20300831_2178429.xml
Unique tags found: ['Activity', 'AnnotatedStopPointRef', 'BankHolidayOperation', 'BoxingDay', 'BoxingDayHoliday', 'ChristmasDay', 'ChristmasDayHoliday', 'ChristmasEve', 'CommonName', 'DateRange', 'DaysOfNonOperation', 'DaysOfOperation', 'DaysOfWeek', 'DepartureTime', 'Description', 'Destination', 'DestinationDisplay', 'Direction', 'Distance', 'EasterMonday', 'EndDate', 'Friday', 'From', 'GoodFriday', 'InboundDescription', 'JourneyCode', 'JourneyPattern', 'JourneyPatternRef', 'JourneyPatternSection', 'JourneyPatternSectionRefs', 'JourneyPatternSections', 'JourneyPatternTimingLink', 'LateSummerBankHolidayNotScotland', 'Latitude', 'LicenceNumber', 'Line', 'LineName', 'LineRef', 'Lines', 'Location', 'Longitude', 'MayDay', 'Monday', 'Name', 'NationalOperatorCode', 'NewYearsDay', 'NewYearsDayHoliday', 'NewYearsEve', 'OperatingPeriod', 'OperatingProfile', 'Operational', 'Operator', 'Opera

## Step 6: Parse Timetable XML — Distributed

One row per `VehicleJourney`, matching trip-level granularity.

In [13]:
timetable_files_rdd = sc.parallelize(timetable_files, numSlices=8)

def parse_timetable_file(iterator):
    rows = []
    for path in iterator:
        try:
            tree = ET.parse(path)
            root = tree.getroot()
        except ET.ParseError:
            continue

        def find(elem, tag):
            for e in elem.iter():
                if e.tag.endswith(tag):
                    return e.text
            return None

        line_name = find(root, "LineName")
        service_code = find(root, "ServiceCode")
        operator = find(root, "OperatorShortName")
        noc = find(root, "NationalOperatorCode")
        origin = find(root, "Origin")
        destination = find(root, "Destination")

        # one row per VehicleJourney (matches trip-level granularity)
        for vj in root.iter():
            if vj.tag.endswith("VehicleJourney"):
                vj_code = find(vj, "VehicleJourneyCode")
                departure = find(vj, "DepartureTime")
                jp_ref = find(vj, "JourneyPatternRef")

                rows.append((
                    line_name, service_code, operator, noc,
                    origin, destination, vj_code, departure, jp_ref,
                    os.path.basename(path)
                ))
    return rows

parsed_timetable_rdd = timetable_files_rdd.mapPartitions(parse_timetable_file)
parsed_timetable_rdd.cache()

tt_row_count = parsed_timetable_rdd.count()
print("Parsed timetable rows (VehicleJourney level):", tt_row_count)
print("Sample rows:", parsed_timetable_rdd.take(3))

[Stage 12:==================================================>       (7 + 1) / 8]

Parsed timetable rows (VehicleJourney level): 41275
Sample rows: [('875', 'PC0003681:18010215', 'Bee Network', 'BNSM', 'The Radclyffe School', 'Hillier Street North', 'vj_1', '15:10:00', 'jp_1', 'BNSM_875_BNSMPC000368118010215875_20250831_20300831_2178429.xml'), ('875', 'PC0003681:18010215', 'Bee Network', 'BNSM', 'The Radclyffe School', 'Hillier Street North', 'vj_2', '07:35:00', 'jp_2', 'BNSM_875_BNSMPC000368118010215875_20250831_20300831_2178429.xml'), ('57', 'PC0003681:18010189', 'Bee Network', 'BNSM', 'Oldham Bus Station', 'Oldham Bus Station', 'vj_1', '15:12:00', 'jp_1', 'BNSM_57_BNSMPC00036811801018957_20260412_20310412_2330237.xml')]


## Step 7: Build the Timetable DataFrame

In [14]:
timetable_schema = StructType([
    StructField("lineName", StringType()),
    StructField("serviceCode", StringType()),
    StructField("operator", StringType()),
    StructField("nationalOperatorCode", StringType()),
    StructField("origin", StringType()),
    StructField("destination", StringType()),
    StructField("vehicleJourneyCode", StringType()),
    StructField("departureTime", StringType()),
    StructField("journeyPatternRef", StringType()),
    StructField("source_file", StringType()),
])

timetable_df = spark.createDataFrame(parsed_timetable_rdd, schema=timetable_schema)
timetable_df = timetable_df.repartition(8, "lineName")
timetable_df.cache()

print("Timetable row count:", timetable_df.count())
print("Distinct lineNames in timetable:", timetable_df.select("lineName").distinct().count())
timetable_df.printSchema()
timetable_df.show(5, truncate=False)

Timetable row count: 41275
Distinct lineNames in timetable: 498
root
 |-- lineName: string (nullable = true)
 |-- serviceCode: string (nullable = true)
 |-- operator: string (nullable = true)
 |-- nationalOperatorCode: string (nullable = true)
 |-- origin: string (nullable = true)
 |-- destination: string (nullable = true)
 |-- vehicleJourneyCode: string (nullable = true)
 |-- departureTime: string (nullable = true)
 |-- journeyPatternRef: string (nullable = true)
 |-- source_file: string (nullable = true)

+--------+------------------+-----------+--------------------+---------------------------------+---------------------------------+------------------+-------------+-----------------+---------------------------------------------------------------+
|lineName|serviceCode       |operator   |nationalOperatorCode|origin                           |destination                      |vehicleJourneyCode|departureTime|journeyPatternRef|source_file                                                 

## Step 8: Check Line Reference Overlap (Location vs Timetable)

Confirms the join key actually works before committing to it.

In [15]:
loc_lines = set(row['lineRef'] for row in location_df.select("lineRef").distinct().collect())
tt_lines = set(row['lineName'] for row in timetable_df.select("lineName").distinct().collect())

print("Distinct lineRefs in location:", len(loc_lines))
print("Distinct lineNames in timetable:", len(tt_lines))

overlap = loc_lines & tt_lines
print("Overlapping lines (will successfully join):", len(overlap))
print("Location lines with NO match in timetable:", len(loc_lines - tt_lines))
print("Sample non-matching location lines:", list(loc_lines - tt_lines)[:15])
print("Sample overlapping lines:", list(overlap)[:15])

Distinct lineRefs in location: 266
Distinct lineNames in timetable: 498
Overlapping lines (will successfully join): 244
Location lines with NO match in timetable: 22
Sample non-matching location lines: ['457', '433', '447', '435', '968', '434', '456', '451', '406', '436', 'SRD1', '458', '450', '403', '468']
Sample overlapping lines: ['378A', '487', '57', '583', '384', '280', '601', '474', '313', '129', '597', '172', '641', '82', '119']


## Step 9: Load the Fares Catalogue

The raw NeTEx fares XML was inspected first, but its structure didn't map cleanly to a per-line table — the pre-built `fares_data_catalogue.csv` catalogue is used instead as the fares reference table.

In [17]:
fares_catalogue_path = os.path.join(PATHS["disruption"], "fares_data_catalogue.csv")

fares_df = spark.read.csv(fares_catalogue_path, header=True, inferSchema=True)

print("Fares catalogue row count:", fares_df.count())
fares_df.printSchema()
fares_df.select("Organisation Name", "National Operator Code", "Line Name", "TariffBasis", "ProductType", "ProductName").show(10, truncate=False)

# check overlap with our lineName join key
fares_lines = set(row['Line Name'] for row in fares_df.select("Line Name").distinct().collect() if row['Line Name'] is not None)
print("\nDistinct Line Names in fares catalogue:", len(fares_lines))

overlap_with_timetable = fares_lines & tt_lines
print("Overlap with timetable lineNames:", len(overlap_with_timetable))

Fares catalogue row count: 121497
root
 |-- Dataset ID: integer (nullable = true)
 |-- XML file name: string (nullable = true)
 |-- Organisation Name: string (nullable = true)
 |-- National Operator Code: string (nullable = true)
 |-- Operator ID: integer (nullable = true)
 |-- BODS Compliant: boolean (nullable = true)
 |-- Last updated date: timestamp (nullable = true)
 |-- Valid from: date (nullable = true)
 |-- Valid to: date (nullable = true)
 |-- Line ids: string (nullable = true)
 |-- Line Name: string (nullable = true)
 |-- ATCO Area: string (nullable = true)
 |-- TariffBasis: string (nullable = true)
 |-- ProductType: string (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- UserType: string (nullable = true)
 |-- Multioperator: string (nullable = true)

+-------------------+----------------------+---------+-----------+-----------+---------------------+
|Organisation Name  |National Operator Code|Line Name|TariffBasis|ProductType|ProductName          |
+---------

## Step 10: Filter Fares to Bee Network Operators

In [18]:
from pyspark.sql.functions import col

BEE_NETWORK_NOCS = ["BNDB", "BNGN", "BNML", "BNSM", "BNVB", "BNFM"]

fares_bn_df = fares_df.filter(col("National Operator Code").isin(BEE_NETWORK_NOCS))

print("Bee Network fares rows (before filter: 121,497):", fares_bn_df.count())
fares_bn_df.groupBy("National Operator Code").count().show()

# recheck line name overlap now that we've filtered to relevant operators
fares_bn_lines = set(row['Line Name'] for row in fares_bn_df.select("Line Name").distinct().collect() if row['Line Name'] is not None)
print("Distinct Line Names (Bee Network only):", len(fares_bn_lines))
print("Overlap with timetable lineNames:", len(fares_bn_lines & tt_lines))

Bee Network fares rows (before filter: 121,497): 4345
+----------------------+-----+
|National Operator Code|count|
+----------------------+-----+
|                  BNGN| 4182|
|                  BNFM|   40|
|                  BNDB|   46|
|                  BNVB|   36|
|                  BNSM|   41|
+----------------------+-----+

Distinct Line Names (Bee Network only): 1
Overlap with timetable lineNames: 1


## Step 11: Build Operator-Level Fares Summary — Merge Strategy

Fares are aggregated by **National Operator Code**, not by line — the raw fares structure is organised around fare zones/operators, so a line-level join would invent a relationship the data doesn't contain. This is the table fares gets joined on later.

In [20]:
from pyspark.sql.functions import first, count as spark_count

fares_operator_summary = fares_bn_df.groupBy("National Operator Code").agg(
    first("Organisation Name").alias("fare_organisation"),
    first("ProductType").alias("common_product_type"),
    first("TariffBasis").alias("common_tariff_basis"),
    first("ProductName").alias("common_product_name"),
    spark_count("*").alias("fare_product_count")
)

fares_operator_summary = fares_operator_summary.withColumnRenamed("National Operator Code", "nationalOperatorCode")

print("Operator-level fares summary:")
fares_operator_summary.show(truncate=False)

Operator-level fares summary:
+--------------------+--------------------------------+-------------------+-------------------+---------------------+------------------+
|nationalOperatorCode|fare_organisation               |common_product_type|common_tariff_basis|common_product_name  |fare_product_count|
+--------------------+--------------------------------+-------------------+-------------------+---------------------+------------------+
|BNDB                |Transport for Greater Manchester|periodPass         |zoneToZone         |1D ALL MODES AD Pass |46                |
|BNFM                |Transport for Greater Manchester|periodPass         |zoneToZone         |1D B+T Z1-4 AD Pass  |40                |
|BNGN                |Transport for Greater Manchester|periodPass         |zoneToZone         |1D B+T Z14 CH OP Pass|4182              |
|BNSM                |Transport for Greater Manchester|periodPass         |zoneToZone         |1D B+T Z14 CH OP Pass|41                |
|BNVB      

## Step 12: Load the Disruption Catalogue

In [21]:
disruption_path = os.path.join(PATHS["disruption"], "disruptions_data_catalogue.csv")

disruption_df = spark.read.csv(disruption_path, header=True, inferSchema=True)

print("Disruption catalogue row count:", disruption_df.count())
disruption_df.printSchema()
disruption_df.show(10, truncate=False)

# check organisation values - do any match "Transport for Greater Manchester" / Bee Network?
disruption_df.select("Organisation").distinct().show(50, truncate=False)

Disruption catalogue row count: 415
root
 |-- Organisation: string (nullable = true)
 |-- ID: string (nullable = true)
 |-- Validity start: string (nullable = true)
 |-- Validity end: string (nullable = true)
 |-- Publication start: string (nullable = true)
 |-- Publication end: string (nullable = true)
 |-- Reason: string (nullable = true)
 |-- Planned: boolean (nullable = true)
 |-- Modes affected: string (nullable = true)
 |-- Operators affected: string (nullable = true)
 |-- Services affected: integer (nullable = true)
 |-- Stops affected: integer (nullable = true)

+---------------+------------------------------------+--------------------------------------------------------------+--------------------------------------------------------------+--------------------------------------------------------------+--------------------------------------------------------------+---------------+-------+--------------+------------------+-----------------+--------------+
|Organisation   |ID      

## Step 13: Filter to TfGM Bus Disruptions & Parse Validity Windows

Filters to bus-mode disruptions from TfGM only, and parses the JS-style date strings into real timestamps.

In [25]:
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")  # needed: 'EEE MMM dd yyyy HH:mm:ss' isn't valid under Spark 3.0+'s default parser

from pyspark.sql.functions import col, to_timestamp, substring, when

# Filter to bus disruptions only (tram rows are irrelevant to Bee Network buses)
bus_disruptions = disruption_df.filter(col("Organisation") == "TfGM") \
                                .filter(col("Modes affected") == "bus")

print("TfGM bus disruption rows:", bus_disruptions.count())

# Parse the JS-style date strings: "Sun Jan 18 2026 18:00:00 GMT+0000 ..."
# Take first 24 chars: "Sun Jan 18 2026 18:00:00" and parse with matching pattern
bus_disruptions = bus_disruptions.withColumn(
    "validity_start_ts",
    to_timestamp(substring(col("Validity start"), 1, 24), "EEE MMM dd yyyy HH:mm:ss")
).withColumn(
    "validity_end_ts",
    to_timestamp(substring(col("Validity end"), 1, 24), "EEE MMM dd yyyy HH:mm:ss")
)

bus_disruptions.select("ID", "Reason", "Planned", "validity_start_ts", "validity_end_ts").show(10, truncate=False)

# how many have a null end date (open-ended / still ongoing)?
bus_disruptions.filter(col("validity_end_ts").isNull()).select("ID", "validity_start_ts").show()

TfGM bus disruption rows: 122
+------------------------------------+---------------+-------+-------------------+-------------------+
|ID                                  |Reason         |Planned|validity_start_ts  |validity_end_ts    |
+------------------------------------+---------------+-------+-------------------+-------------------+
|c3a3f86a-7443-4d9b-accb-3aa40d6b3392|maintenanceWork|true   |2026-01-18 18:00:00|2027-06-17 19:00:00|
|9942b82c-16bf-4d1e-b35e-bbc56c718ba6|maintenanceWork|true   |2026-02-05 00:00:00|2026-02-27 12:05:00|
|b2d96d0e-76e1-4ca7-912e-4884e0cffba6|maintenanceWork|true   |2026-02-09 00:00:00|2027-02-09 23:59:00|
|2cd1b1d5-08ee-440b-90b2-568830611a0e|roadworks      |true   |2026-02-10 13:26:00|2026-08-31 22:59:00|
|72acbf50-4ac3-4158-8da6-52056cdeb454|roadClosed     |false  |2026-02-23 00:00:00|2026-11-22 23:59:00|
|ba17eaca-45f6-4cd4-8f4a-52b648bfd679|roadClosed     |true   |2026-02-23 00:00:00|2026-11-30 23:59:00|
|3f9c934e-1c0c-4e8e-ace4-66aa2510d613|route

## Step 14: Build the Final Disruption Lookup Table

Open-ended disruptions (null end date) are treated as active far into the future rather than dropped.

In [26]:
from pyspark.sql.functions import coalesce, lit

# For open-ended disruptions (null end date), treat as still active far into the future
bus_disruptions_clean = bus_disruptions.withColumn(
    "validity_end_ts_filled",
    coalesce(col("validity_end_ts"), to_timestamp(lit("2030-01-01 00:00:00")))
)

# keep only the columns we need for the join
disruption_lookup = bus_disruptions_clean.select(
    col("ID").alias("disruption_id"),
    "Reason",
    "Planned",
    "validity_start_ts",
    col("validity_end_ts_filled").alias("validity_end_ts")
)

disruption_lookup.cache()
print("Disruption lookup table rows:", disruption_lookup.count())
disruption_lookup.show(10, truncate=False)

Disruption lookup table rows: 122
+------------------------------------+---------------+-------+-------------------+-------------------+
|disruption_id                       |Reason         |Planned|validity_start_ts  |validity_end_ts    |
+------------------------------------+---------------+-------+-------------------+-------------------+
|c3a3f86a-7443-4d9b-accb-3aa40d6b3392|maintenanceWork|true   |2026-01-18 18:00:00|2027-06-17 19:00:00|
|9942b82c-16bf-4d1e-b35e-bbc56c718ba6|maintenanceWork|true   |2026-02-05 00:00:00|2026-02-27 12:05:00|
|b2d96d0e-76e1-4ca7-912e-4884e0cffba6|maintenanceWork|true   |2026-02-09 00:00:00|2027-02-09 23:59:00|
|2cd1b1d5-08ee-440b-90b2-568830611a0e|roadworks      |true   |2026-02-10 13:26:00|2026-08-31 22:59:00|
|72acbf50-4ac3-4158-8da6-52056cdeb454|roadClosed     |false  |2026-02-23 00:00:00|2026-11-22 23:59:00|
|ba17eaca-45f6-4cd4-8f4a-52b648bfd679|roadClosed     |true   |2026-02-23 00:00:00|2026-11-30 23:59:00|
|3f9c934e-1c0c-4e8e-ace4-66aa2510d613|r

## Step 15: The Final Combined Join — First Attempt

🐛 **Bug:** this produced **2.48 billion rows** — an uncontrolled fan-out. Location is left-joined to timetable on line reference, to the fares summary on operator code, and to disruptions on a time-range overlap (not equality).

In [27]:
from pyspark.sql.functions import broadcast, col

# 1) location (big) LEFT JOIN timetable (small) on line code
combined_df = location_df.join(
    broadcast(timetable_df),
    location_df["lineRef"] == timetable_df["lineName"],
    how="left"
)

# 2) LEFT JOIN fares summary on operator code
combined_df = combined_df.join(
    broadcast(fares_operator_summary),
    combined_df["nationalOperatorCode"] == fares_operator_summary["nationalOperatorCode"],
    how="left"
).drop(fares_operator_summary["nationalOperatorCode"])  # avoid duplicate column name

# 3) LEFT JOIN disruption on time-range overlap (not equality)
combined_df = combined_df.join(
    broadcast(disruption_lookup),
    (combined_df["timestamp"] >= disruption_lookup["validity_start_ts"]) &
    (combined_df["timestamp"] <= disruption_lookup["validity_end_ts"]),
    how="left"
)

combined_df.cache()

final_row_count = combined_df.count()
print("✅ FINAL COMBINED ROW COUNT:", final_row_count)
print("Meets 100k requirement:", final_row_count >= 100000)

combined_df.printSchema()
combined_df.show(5, truncate=False)

26/07/23 07:55:34 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/07/23 07:56:09 WARN MemoryStore: Not enough space to cache rdd_274_0 in memory! (computed 283.9 MiB so far)
26/07/23 07:56:10 WARN BlockManager: Persisting block rdd_274_0 to disk instead.
26/07/23 07:56:10 WARN MemoryStore: Not enough space to cache rdd_274_4 in memory! (computed 303.4 MiB so far)
26/07/23 07:56:10 WARN BlockManager: Persisting block rdd_274_4 to disk instead.
26/07/23 07:56:24 WARN MemoryStore: Not enough space to cache rdd_274_1 in memory! (computed 380.6 MiB so far)
26/07/23 07:56:24 WARN BlockManager: Persisting block rdd_274_1 to disk instead.
26/07/23 07:56:42 WARN MemoryStore: Not enough space to cache rdd_274_6 in memory! (computed 360.4 MiB so far)
26/07/23 07:56:42 WARN BlockManager: Persisting block rdd_274_6 to disk instead.
26/07/23 07:56:52 WARN MemoryStore: Not eno

✅ FINAL COMBINED ROW COUNT: 2484047294
Meets 100k requirement: True
root
 |-- timestamp: timestamp (nullable = true)
 |-- lineRef: string (nullable = true)
 |-- directionRef: string (nullable = true)
 |-- vehicleRef: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- source_file: string (nullable = true)
 |-- lineName: string (nullable = true)
 |-- serviceCode: string (nullable = true)
 |-- operator: string (nullable = true)
 |-- nationalOperatorCode: string (nullable = true)
 |-- origin: string (nullable = true)
 |-- destination: string (nullable = true)
 |-- vehicleJourneyCode: string (nullable = true)
 |-- departureTime: string (nullable = true)
 |-- journeyPatternRef: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- fare_organisation: string (nullable = true)
 |-- common_product_type: string (nullable = true)
 |-- common_tariff_basis: string (nullable = true)
 |-- common_product_name: string (nullab

[Stage 108:>                                                        (0 + 1) / 1]

+-------------------+-------+------------+----------+---------+---------+------------------------------+--------+------------------+-----------+--------------------+-----------------------------+-----------------------+------------------+-------------+-----------------+---------------------------------------------------------------+--------------------------------+-------------------+-------------------+---------------------+------------------+------------------------------------+---------------+-------+-------------------+-------------------+
|timestamp          |lineRef|directionRef|vehicleRef|latitude |longitude|source_file                   |lineName|serviceCode       |operator   |nationalOperatorCode|origin                       |destination            |vehicleJourneyCode|departureTime|journeyPatternRef|source_file                                                    |fare_organisation               |common_product_type|common_tariff_basis|common_product_name  |fare_product_count|di

26/07/23 08:36:04 WARN MemoryStore: Not enough space to cache rdd_274_0 in memory! (computed 1741.4 MiB so far)
                                                                                

## Step 16: Diagnosing the Fan-Out

The timetable has many duplicate journey registrations per line — every location ping was multiplying against every duplicate.

In [29]:
# how many timetable rows does each lineRef/lineName actually have?
line_match_counts = timetable_df.groupBy("lineName").count().orderBy(col("count").desc())
line_match_counts.show(20, truncate=False)

# cross-check: for the top offending lines, are journeys genuinely distinct or duplicated?
top_line = line_match_counts.first()["lineName"]
print(f"\nInspecting line: {top_line}")
timetable_df.filter(col("lineName") == top_line).select(
    "serviceCode", "vehicleJourneyCode", "departureTime", "source_file"
).show(20, truncate=False)

+--------+-----+
|lineName|count|
+--------+-----+
|192     |1592 |
|17      |1084 |
|250     |663  |
|135     |588  |
|143     |552  |
|219     |552  |
|203     |529  |
|43      |528  |
|V1      |528  |
|142     |508  |
|18      |494  |
|582     |489  |
|11      |474  |
|216     |460  |
|84      |450  |
|50      |445  |
|86      |445  |
|33      |444  |
|83      |441  |
|201     |438  |
+--------+-----+
only showing top 20 rows

Inspecting line: 192
+------------------+------------------+-------------+---------------------------------------------------------------+
|serviceCode       |vehicleJourneyCode|departureTime|source_file                                                    |
+------------------+------------------+-------------+---------------------------------------------------------------+
|PC0003681:18010530|vj_1              |04:05:00     |BNSM_192_BNSMPC000368118010530192_20260102_20310102_2267048.xml|
|PC0003681:18010530|vj_2              |04:25:00     |BNSM_192_BNSMPC00036

## Step 17: The Fix — Cap Journeys Per Line

Capping to 10 representative journeys per line keeps schedule diversity while controlling the fan-out.

In [30]:
from pyspark.sql import Window
from pyspark.sql.functions import row_number

# cap to a maximum of 10 representative journeys per line (keeps schedule diversity, controls fan-out)
w = Window.partitionBy("lineName").orderBy("departureTime")

timetable_capped = timetable_df.withColumn("rn", row_number().over(w)) \
                                .filter(col("rn") <= 10) \
                                .drop("rn")

print("Timetable rows before cap:", timetable_df.count())
print("Timetable rows after cap (max 10/line):", timetable_capped.count())

Timetable rows before cap: 41275
Timetable rows after cap (max 10/line): 3048


## Step 18: Final Combined Join (Location + Timetable + Fares)

✅ Row count drops from 2.48 billion to the correct **771,733**.

In [31]:
combined_df = location_df.join(
    broadcast(timetable_capped),
    location_df["lineRef"] == timetable_capped["lineName"],
    how="left"
)

combined_df = combined_df.join(
    broadcast(fares_operator_summary),
    combined_df["nationalOperatorCode"] == fares_operator_summary["nationalOperatorCode"],
    how="left"
).drop(fares_operator_summary["nationalOperatorCode"])

combined_df = combined_df.repartition(8, "lineRef")
combined_df.cache()

capped_row_count = combined_df.count()
print("Row count after capped join (location+timetable+fares):", capped_row_count)
combined_df.show(5, truncate=False)

Row count after capped join (location+timetable+fares): 771733
+-------------------+-------+------------+----------+---------+---------+------------------------------+--------+------------------+-----------+--------------------+-----------------------------+-----------------------+------------------+-------------+-----------------+---------------------------------------------------------------+--------------------------------+-------------------+-------------------+---------------------+------------------+
|timestamp          |lineRef|directionRef|vehicleRef|latitude |longitude|source_file                   |lineName|serviceCode       |operator   |nationalOperatorCode|origin                       |destination            |vehicleJourneyCode|departureTime|journeyPatternRef|source_file                                                    |fare_organisation               |common_product_type|common_tariff_basis|common_product_name  |fare_product_count|
+-------------------+-------+----------

## Step 19: Disruption Exposure — First Attempt (Broken Flag)

🐛 **Bug:** a true/false disruption flag, computed via a broadcast variable to avoid duplicating rows — but it came back true for **every single record**, giving zero variance.

In [32]:
from pyspark.sql.functions import udf, col
from pyspark.sql.types import BooleanType, StringType
from datetime import datetime

# collect small disruption table to driver - safe, only 122 rows
disruption_rows = disruption_lookup.select("validity_start_ts", "validity_end_ts", "Reason").collect()
disruption_windows = [(r["validity_start_ts"], r["validity_end_ts"], r["Reason"]) for r in disruption_rows]

# broadcast this small list to all executors
broadcast_disruptions = sc.broadcast(disruption_windows)

def check_disruption(ts):
    if ts is None:
        return False
    for start, end, reason in broadcast_disruptions.value:
        if start is not None and end is not None and start <= ts <= end:
            return True
    return False

def get_disruption_reason(ts):
    if ts is None:
        return None
    for start, end, reason in broadcast_disruptions.value:
        if start is not None and end is not None and start <= ts <= end:
            return reason
    return None

disruption_flag_udf = udf(check_disruption, BooleanType())
disruption_reason_udf = udf(get_disruption_reason, StringType())

combined_df = combined_df.withColumn("disruption_active", disruption_flag_udf(col("timestamp")))
combined_df = combined_df.withColumn("disruption_reason", disruption_reason_udf(col("timestamp")))

combined_df.cache()
final_count = combined_df.count()
print("Final row count (with disruption flag, no fan-out):", final_count)

combined_df.groupBy("disruption_active").count().show()
combined_df.select("timestamp", "lineRef", "disruption_active", "disruption_reason").show(10, truncate=False)

[Stage 157:=================================================>       (7 + 1) / 8]

Final row count (with disruption flag, no fan-out): 771733
+-----------------+------+
|disruption_active| count|
+-----------------+------+
|             true|771733|
+-----------------+------+

+-------------------+-------+-----------------+-----------------+
|timestamp          |lineRef|disruption_active|disruption_reason|
+-------------------+-------+-----------------+-----------------+
|2026-07-22 20:28:47|191    |true             |maintenanceWork  |
|2026-07-22 20:28:47|191    |true             |maintenanceWork  |
|2026-07-22 20:28:47|191    |true             |maintenanceWork  |
|2026-07-22 20:28:47|191    |true             |maintenanceWork  |
|2026-07-22 20:28:47|191    |true             |maintenanceWork  |
|2026-07-22 20:28:47|191    |true             |maintenanceWork  |
|2026-07-22 20:28:47|191    |true             |maintenanceWork  |
|2026-07-22 20:28:47|191    |true             |maintenanceWork  |
|2026-07-22 20:28:47|191    |true             |maintenanceWork  |
|2026-07-22 2

## Step 20: Diagnosing the Broken Flag

Checking the duration distribution of disruptions reveals several span the entire data-collection window, which is why the flag was true for every ping.

In [34]:
disruption_lookup_with_duration.select("disruption_id", "Reason", "duration_days").orderBy(col("duration_days").asc()).show(20, truncate=False)

# summary stats
disruption_lookup_with_duration.select("duration_days").describe().show()

+------------------------------------+---------------+-------------+
|disruption_id                       |Reason         |duration_days|
+------------------------------------+---------------+-------------+
|cc51b6a4-656e-479d-9875-67fad839a8e2|congestion     |0            |
|d037fbb8-add3-40aa-ae5b-754c1ed27f13|specialEvent   |0            |
|902c0a01-e0d5-4839-bf80-8bf897dad3fa|specialEvent   |0            |
|dd1416ce-9600-46d1-bae5-5e5132692f3c|specialEvent   |0            |
|c1dcd983-f8a1-4a22-b1df-637af61abf0e|roadworks      |1            |
|3cb3de1d-7a91-49c8-a0b1-80f5087d63be|roadworks      |2            |
|bc13a8e6-5b8b-46dd-8d8b-8594834b0902|maintenanceWork|2            |
|709cc41c-be97-490f-95dd-964b7aebe6a9|maintenanceWork|2            |
|58e24459-ee0b-4a22-b2b3-ed7014308fb8|maintenanceWork|2            |
|51d1f554-3bec-4544-845e-404c0c1abc4d|maintenanceWork|2            |
|5cd9d901-7862-468c-916f-80e58db69e3b|maintenanceWork|2            |
|5db53082-582a-437f-8859-d043dbf05

## Step 21: The Fix — Filter to Short-Term Disruptions

Restricting to disruptions lasting 14 days or less removes the always-active long-running ones and rebuilds the broadcast lookup from just the short, genuinely discriminating disruptions.

In [35]:
from pyspark.sql.functions import datediff

# recompute duration and filter to short-term, genuinely discriminating disruptions
disruption_lookup_with_duration = disruption_lookup.withColumn(
    "duration_days", datediff(col("validity_end_ts"), col("validity_start_ts"))
)

short_disruptions = disruption_lookup_with_duration.filter(col("duration_days") <= 14)

print("Short-term disruptions (<=14 days):", short_disruptions.count())
short_disruptions.select("disruption_id", "Reason", "duration_days").orderBy("duration_days").show(30, truncate=False)

# rebuild broadcast list using only these
short_disruption_rows = short_disruptions.select("validity_start_ts", "validity_end_ts", "Reason").collect()
short_disruption_windows = [(r["validity_start_ts"], r["validity_end_ts"], r["Reason"]) for r in short_disruption_rows]

broadcast_short_disruptions = sc.broadcast(short_disruption_windows)

def check_disruption_v2(ts):
    if ts is None:
        return False
    for start, end, reason in broadcast_short_disruptions.value:
        if start is not None and end is not None and start <= ts <= end:
            return True
    return False

def get_disruption_reason_v2(ts):
    if ts is None:
        return None
    for start, end, reason in broadcast_short_disruptions.value:
        if start is not None and end is not None and start <= ts <= end:
            return reason
    return None

disruption_flag_udf_v2 = udf(check_disruption_v2, BooleanType())
disruption_reason_udf_v2 = udf(get_disruption_reason_v2, StringType())

# drop old columns, apply corrected ones
combined_df = combined_df.drop("disruption_active", "disruption_reason")
combined_df = combined_df.withColumn("disruption_active", disruption_flag_udf_v2(col("timestamp")))
combined_df = combined_df.withColumn("disruption_reason", disruption_reason_udf_v2(col("timestamp")))

combined_df.cache()
print("Final row count:", combined_df.count())
combined_df.groupBy("disruption_active").count().show()

Short-term disruptions (<=14 days): 78
+------------------------------------+---------------+-------------+
|disruption_id                       |Reason         |duration_days|
+------------------------------------+---------------+-------------+
|cc51b6a4-656e-479d-9875-67fad839a8e2|congestion     |0            |
|d037fbb8-add3-40aa-ae5b-754c1ed27f13|specialEvent   |0            |
|902c0a01-e0d5-4839-bf80-8bf897dad3fa|specialEvent   |0            |
|dd1416ce-9600-46d1-bae5-5e5132692f3c|specialEvent   |0            |
|c1dcd983-f8a1-4a22-b1df-637af61abf0e|roadworks      |1            |
|678c166d-90b6-4c5b-9ed9-e60045a63085|maintenanceWork|2            |
|3cb3de1d-7a91-49c8-a0b1-80f5087d63be|roadworks      |2            |
|bc13a8e6-5b8b-46dd-8d8b-8594834b0902|maintenanceWork|2            |
|709cc41c-be97-490f-95dd-964b7aebe6a9|maintenanceWork|2            |
|58e24459-ee0b-4a22-b2b3-ed7014308fb8|maintenanceWork|2            |
|51d1f554-3bec-4544-845e-404c0c1abc4d|maintenanceWork|2         

[Stage 181:=================================================>       (7 + 1) / 8]

Final row count: 771733
+-----------------+------+
|disruption_active| count|
+-----------------+------+
|             true|771733|
+-----------------+------+



## Step 22: From Flag to Count — Final Disruption Exposure Feature

✅ Converting to a **count** of overlapping disruptions (instead of a true/false flag) restores real variance (range 47–53) while keeping the row count unchanged at 771,733 — no duplication.

In [37]:
def count_disruptions(ts):
    if ts is None:
        return 0
    count = 0
    for start, end, reason in broadcast_short_disruptions.value:
        if start is not None and end is not None and start <= ts <= end:
            count += 1
    return count

from pyspark.sql.types import IntegerType
disruption_count_udf = udf(count_disruptions, IntegerType())

combined_df = combined_df.drop("disruption_active", "disruption_reason")
combined_df = combined_df.withColumn("disruption_count", disruption_count_udf(col("timestamp")))

combined_df.cache()
print("Final row count:", combined_df.count())
combined_df.select("disruption_count").describe().show()
combined_df.groupBy("disruption_count").count().orderBy("disruption_count").show()

Final row count: 771733
+-------+------------------+
|summary|  disruption_count|
+-------+------------------+
|  count|            771733|
|   mean| 51.88175573676388|
| stddev|0.8457992641485589|
|    min|                47|
|    max|                53|
+-------+------------------+

+----------------+------+
|disruption_count| count|
+----------------+------+
|              47|  3261|
|              48|  6886|
|              49| 34881|
|              52|669466|
|              53| 57239|
+----------------+------+



## Step 23: Final Combined Dataset Summary

In [38]:
print("=== FINAL COMBINED RAW DATASET SUMMARY ===")
print("Total rows:", combined_df.count())
print("Total columns:", len(combined_df.columns))
print("Columns:", combined_df.columns)
print("\nPartitions:", combined_df.rdd.getNumPartitions())

# quick null check across key columns
from pyspark.sql.functions import col, sum as spark_sum, when

combined_df.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) 
    for c in ["timestamp", "lineRef", "latitude", "longitude", "lineName", "nationalOperatorCode", "fare_organisation", "disruption_count"]
]).show(truncate=False)

=== FINAL COMBINED RAW DATASET SUMMARY ===
Total rows: 771733
Total columns: 23
Columns: ['timestamp', 'lineRef', 'directionRef', 'vehicleRef', 'latitude', 'longitude', 'source_file', 'lineName', 'serviceCode', 'operator', 'nationalOperatorCode', 'origin', 'destination', 'vehicleJourneyCode', 'departureTime', 'journeyPatternRef', 'source_file', 'fare_organisation', 'common_product_type', 'common_tariff_basis', 'common_product_name', 'fare_product_count', 'disruption_count']

Partitions: 8
+---------+-------+--------+---------+--------+--------------------+-----------------+----------------+
|timestamp|lineRef|latitude|longitude|lineName|nationalOperatorCode|fare_organisation|disruption_count|
+---------+-------+--------+---------+--------+--------------------+-----------------+----------------+
|0        |0      |0       |0        |2423    |2423                |244893           |0               |
+---------+-------+--------+---------+--------+--------------------+-----------------+----

## Step 24: Investigating Missing Fares Matches (BNML)

244,893 rows have no fares match. This confirms the cause: **BNML** is a valid operator in the timetable data but simply has no entry in the fares catalogue — a genuine gap in the source data, not a join bug.

In [40]:
from pyspark.sql.functions import expr
# how many rows have a nationalOperatorCode but STILL got no fares match?
combined_df.filter(col("nationalOperatorCode").isNotNull() & col("fare_organisation").isNull()) \
    .select("nationalOperatorCode").distinct().show()

# compare to what's actually in fares_operator_summary
fares_operator_summary.select("nationalOperatorCode").show()

# check for whitespace/case mismatches
combined_df.filter(col("nationalOperatorCode").isNotNull() & col("fare_organisation").isNull()) \
    .select("nationalOperatorCode").distinct() \
    .withColumn("code_length", expr("length(nationalOperatorCode)")) \
    .show()

+--------------------+
|nationalOperatorCode|
+--------------------+
|                BNML|
+--------------------+

+--------------------+
|nationalOperatorCode|
+--------------------+
|                BNGN|
|                BNFM|
|                BNDB|
|                BNVB|
|                BNSM|
+--------------------+

+--------------------+-----------+
|nationalOperatorCode|code_length|
+--------------------+-----------+
|                BNML|          4|
+--------------------+-----------+



## Step 25: Export the Final Combined Raw Dataset

In [42]:
import os

output_dir = os.path.join(PROJECT_ROOT, "data/processed")
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "combined_raw_dataset.csv")

combined_df.coalesce(4).write.mode("overwrite").option("header", "true").csv(output_path)

print(f"✅ Exported to: {output_path}")

# quick verification - list what got written
import glob
written_files = glob.glob(os.path.join(output_path, "*.csv"))
print(f"Number of part-files written: {len(written_files)}")
for f in written_files:
    print(" -", os.path.basename(f), f"({os.path.getsize(f)/1024/1024:.1f} MB)")

[Stage 250:>                                                        (0 + 4) / 4]

✅ Exported to: /Users/aayushbohara/Desktop/smartcity-bus-clustering/data/processed/combined_raw_dataset.csv
Number of part-files written: 4
 - part-00003-b122674e-82a0-4446-b26b-96a27a58b3af-c000.csv (50.6 MB)
 - part-00002-b122674e-82a0-4446-b26b-96a27a58b3af-c000.csv (75.4 MB)
 - part-00001-b122674e-82a0-4446-b26b-96a27a58b3af-c000.csv (47.2 MB)
 - part-00000-b122674e-82a0-4446-b26b-96a27a58b3af-c000.csv (63.9 MB)
